In [2]:
import torch
from torch import nn
from typing import Optional


1/torch.logspace(start=torch.log10(torch.tensor(0.1)), 
                 end=torch.log10(torch.tensor(20)), steps=10)

tensor([10.0000,  5.5505,  3.0808,  1.7100,  0.9491,  0.5268,  0.2924,  0.1623,
         0.0901,  0.0500])

In [28]:
import sys
from pathlib import Path

project_root = Path.cwd()
sys.path.insert(0, str(project_root / "scripts"))

from torch.utils.data import DataLoader
from data import CanonicalParquetDataset, CANONICAL_MULTIPLICITIES
from model import FoundationNMRProcessor, H_AVAILABILITY_FIELDS, MAX_J_VALUES, MULTIPLICITY_TO_ID
from scripts.model.utils.fourier_features import FourierFeatures, RBFExpansion

dataset_path = project_root / "datasets/cleaned/test_benchmark.parquet"
assert dataset_path.is_file(), f"Dataset non trovato: {dataset_path}"

dataset = CanonicalParquetDataset(dataset_path)
processor = FoundationNMRProcessor()

loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    collate_fn=processor,
)

batch = next(iter(loader))

print("Record IDs:")
print(batch["record_ids"])

print("\nTensor shapes:")
for modality in ("h", "c"):
    for name, tensor in batch[modality].items():
        print(f"{modality}.{name:18s} {tuple(tensor.shape)}  {tensor.dtype}")

print("\nNumero di picchi reali per record:")
print("H:", batch["h"]["peak_mask"].sum(dim=1).tolist())
print("C:", batch["c"]["peak_mask"].sum(dim=1).tolist())

print("\nAvailability H:")
for channel, name in enumerate(H_AVAILABILITY_FIELDS):
    available = batch["h"]["availability"][:, :, channel].sum(dim=1)
    print(f"{name:18s}: {available.tolist()}")

# Ispeziona i valori reali del primo record.
h_mask = batch["h"]["peak_mask"][0]
c_mask = batch["c"]["peak_mask"][0]

print("\nPrimo record:", batch["record_ids"][0])
print("H shifts:", batch["h"]["shift"][0][h_mask].tolist())
print("C shifts:", batch["c"]["shift"][0][c_mask].tolist())
print("H integration:", batch["h"]["integration"][0][h_mask].tolist())
print("H multiplicity IDs:", batch["h"]["multiplicity"][0][h_mask].tolist())
print("H range half-span:", batch["h"]["range_half_span"][0][h_mask].tolist())

assert batch["h"]["shift"].shape == (8, 60)
assert batch["h"]["j_values"].shape == (8, 60, 6)
assert batch["h"]["availability"].shape == (8, 60, 4)
assert batch["c"]["shift"].shape == (8, 60)

print("\n✓ Primo batch reale processato correttamente")

Record IDs:
['nmrpeak-mst-nmr:test:10034', 'nmrpeak-mst-nmr:test:10037', 'nmrpeak-mst-nmr:test:10044', 'nmrpeak-mst-nmr:test:10051', 'nmrpeak-mst-nmr:test:1008', 'nmrpeak-mst-nmr:test:10085', 'nmrpeak-mst-nmr:test:1009', 'nmrpeak-mst-nmr:test:10112']

Tensor shapes:
h.shift              (8, 60)  torch.float32
h.integration        (8, 60)  torch.float32
h.multiplicity       (8, 60)  torch.int64
h.j_values           (8, 60, 6)  torch.float32
h.range_half_span    (8, 60)  torch.float32
h.peak_mask          (8, 60)  torch.bool
h.j_mask             (8, 60, 6)  torch.bool
h.availability       (8, 60, 4)  torch.bool
c.shift              (8, 60)  torch.float32
c.peak_mask          (8, 60)  torch.bool

Numero di picchi reali per record:
H: [7, 5, 12, 9, 13, 15, 9, 14]
C: [16, 10, 24, 27, 17, 18, 12, 22]

Availability H:
integration       : [7, 5, 12, 9, 13, 15, 9, 14]
multiplicity      : [7, 5, 12, 9, 13, 15, 9, 14]
j_values          : [7, 5, 12, 9, 13, 15, 9, 14]
range_half_span   : [7, 5, 12,

In [4]:
H_AVAILABILITY_FIELDS

('integration', 'multiplicity', 'j_values', 'range_half_span')

In [5]:
batch['h'].keys()

dict_keys(['shift', 'integration', 'multiplicity', 'j_values', 'range_half_span', 'peak_mask', 'j_mask', 'availability'])

In [6]:
batch['c'].keys()

dict_keys(['shift', 'peak_mask'])

In [7]:
_, h_peaks, c_peaks = batch.values()
h_shift, h_integration, h_multiplicity, h_j_values, h_range_half_span, h_peak_mask, h_j_mask, h_availability = h_peaks.values()

In [8]:
rel_integration = (h_integration / torch.sum(h_integration, dim=1).unsqueeze(-1))
log1p_integration = torch.log1p(h_integration)
log1p_integration.shape, rel_integration.shape

(torch.Size([8, 60]), torch.Size([8, 60]))

In [9]:
h_j_values.shape, h_j_mask.shape

(torch.Size([8, 60, 6]), torch.Size([8, 60, 6]))

In [10]:
torch.concat( [log1p_integration.unsqueeze(-1), rel_integration.unsqueeze(-1)], dim=-1 ).shape

torch.Size([8, 60, 2])

In [11]:
a_i, a_m, a_j, a_w = torch.unbind(h_availability, dim=-1)

In [12]:
embeddings = torch.ones((8, 60, 64))
a = h_availability[..., 0]
(a.unsqueeze(-1) * embeddings).shape

torch.Size([8, 60, 64])

In [13]:
embeddings_i = torch.ones((8, 60, 64))
embeddings_m = torch.ones((8, 60, 32))
embeddings_w = torch.ones((8, 60, 32))

torch.concat((embeddings_i, embeddings_m, embeddings_w, embeddings_w, a_i.unsqueeze(-1), a_m.unsqueeze(-1), a_w.unsqueeze(-1)), dim=-1).shape

torch.Size([8, 60, 163])

In [14]:
h_availability.shape

torch.Size([8, 60, 4])

In [15]:
rbf = RBFExpansion(
    x_min=0,
    x_max=10,
    n=16,
    sigma=1.0,
)

j_value_mlp = nn.Sequential(
        nn.Linear(16+1, 64),
        nn.GELU(),
        nn.Linear(64, 64),
    )

In [16]:
torch.log1p(h_j_values).unsqueeze(-1).shape

torch.Size([8, 60, 6, 1])

In [17]:
features = rbf(h_j_values)

print(features.shape)  # torch.Size([3, 16])

features = torch.concat( [ features,  torch.log1p(h_j_values).unsqueeze(-1)], dim=-1 )

print(features.shape)  # torch.Size([3, 16])

features_mlp = j_value_mlp(features)

print(features_mlp.shape)  # torch.Size([3, 16])

features_mlp_gated = h_j_mask.unsqueeze(-1) * features_mlp

print(features_mlp_gated.shape)  # torch.Size([3, 16])

pooled = torch.sum(features_mlp_gated, dim=2)

print(pooled.shape)  # torch.Size([3, 16])

k_relative = torch.sum(h_j_mask, dim=-1)/MAX_J_VALUES

pooled = torch.concat([ pooled,  k_relative.unsqueeze(-1) ], dim=-1)

print(pooled.shape)  # torch.Size([3, 16])


torch.Size([8, 60, 6, 16])
torch.Size([8, 60, 6, 17])
torch.Size([8, 60, 6, 64])
torch.Size([8, 60, 6, 64])
torch.Size([8, 60, 64])
torch.Size([8, 60, 65])


In [18]:
k_relative = torch.sum(h_j_mask, dim=-1)/MAX_J_VALUES

torch.concat([ pooled,  k_relative.unsqueeze(-1) ], dim=-1).shape

torch.Size([8, 60, 66])

In [ ]:
class ShiftEmbedder(nn.Module):
    def __init__(
        self,
        x_min,
        x_max,
        d_model,
        fourier_strategy,
        num_fourier_freqs,
    ):
        super().__init__()
        self.fourier = FourierFeatures(
            strategy=fourier_strategy,
            x_min=x_min,
            x_max=x_max,
            num_freqs=num_fourier_freqs,
        )
        self.mlp = nn.Sequential(
            nn.Linear(self.fourier.num_features(), d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, d_model),
        )

    def forward(self, shift):
        return self.mlp(self.fourier(shift.unsqueeze(-1)))


class IntegrationEmbedder(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(2, 32),
            nn.GELU(),
        )

    def forward(self, integration):
        total_integration = integration.sum(dim=1, keepdim=True).clamp_min(1)
        relative_integration = integration / total_integration
        log1p_integration = torch.log1p(integration)
        integration_vector = torch.concat(
            [log1p_integration.unsqueeze(-1), relative_integration.unsqueeze(-1)],
            dim=-1,
        )
        return self.mlp(integration_vector)


class MultiplicityEmbedder(nn.Module):
    def __init__(self, num_multiplicities):
        super().__init__()
        self.multiplicity_embeddor = nn.Embedding(num_multiplicities, 32)

    def forward(self, multiplicities):
        return self.multiplicity_embeddor(multiplicities)


class JCouplingEmbedder(nn.Module):
    def __init__(self, j_min, j_max, num_rbf_centers, sigma):
        super().__init__()
        self.rbf_embeddor = RBFExpansion(
            n=num_rbf_centers,
            x_min=j_min,
            x_max=j_max,
            sigma=sigma,
        )
        self.j_value_mlp = nn.Sequential(
            nn.Linear(num_rbf_centers + 1, 64),
            nn.GELU(),
            nn.Linear(64, 64),
        )
        self.j_pool_proj = nn.Sequential(
            nn.Linear(65, 64),
            nn.GELU(),
        )

    def forward(self, j_values, j_mask):
        j_vector = self.rbf_embeddor(j_values)
        j_vector = torch.concat(
            [j_vector, torch.log1p(j_values).unsqueeze(-1)],
            dim=-1,
        )
        j_embedd = self.j_value_mlp(j_vector)
        pooled_j_embedd = torch.sum(j_mask.unsqueeze(-1) * j_embedd, dim=2)
        num_j_per_peak = j_mask.sum(dim=-1, dtype=j_embedd.dtype) / MAX_J_VALUES
        pooled_j_embedd = torch.concat(
            [pooled_j_embedd, num_j_per_peak.unsqueeze(-1)],
            dim=-1,
        )
        return self.j_pool_proj(pooled_j_embedd)


class WidthEmbedder(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(1, 32),
            nn.GELU(),
        )

    def forward(self, width):
        return self.mlp(width.unsqueeze(-1))


class HPeakEmbedder(nn.Module):
    def __init__(
        self,
        d_model,
        h_range,
        fourier_strategy,
        num_fourier_freqs,
        num_multiplicities,
        j_min,
        j_max,
        num_rbf_centers,
        rbf_sigma,
    ):
        super().__init__()
        self.shift_embedder = ShiftEmbedder(
            *h_range,
            d_model=d_model,
            fourier_strategy=fourier_strategy,
            num_fourier_freqs=num_fourier_freqs,
        )
        self.integration_embedder = IntegrationEmbedder()
        self.multiplicity_embedder = MultiplicityEmbedder(num_multiplicities)
        self.j_embedder = JCouplingEmbedder(
            j_min,
            j_max,
            num_rbf_centers,
            rbf_sigma,
        )
        self.width_embedder = WidthEmbedder()
        self.rich_mlp = nn.Sequential(
            nn.Linear(164, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, d_model),
        )
        final_layer = self.rich_mlp[-1]
        nn.init.normal_(final_layer.weight, mean=0.0, std=1e-3)
        nn.init.zeros_(final_layer.bias)
        self.layer_norm = nn.LayerNorm(d_model)

    def forward(
        self,
        shift: torch.Tensor,
        integration: Optional[torch.Tensor] = None,
        multiplicity: Optional[torch.Tensor] = None,
        j_values: Optional[torch.Tensor] = None,
        width: Optional[torch.Tensor] = None,
        j_mask: Optional[torch.Tensor] = None,
        availability: Optional[torch.Tensor] = None,
    ):
        availability_features = availability.to(dtype=shift.dtype)
        a_i, a_m, a_j, a_w = torch.unbind(availability_features, dim=-1)
        shift_embedd = self.shift_embedder(shift)
        integration_embedd = a_i.unsqueeze(-1) * self.integration_embedder(integration)
        multiplicity_embedd = a_m.unsqueeze(-1) * self.multiplicity_embedder(multiplicity)
        j_embedd = a_j.unsqueeze(-1) * self.j_embedder(j_values, j_mask)
        width_embedd = a_w.unsqueeze(-1) * self.width_embedder(width)
        rich_input = torch.concat(
            (
                integration_embedd,
                multiplicity_embedd,
                j_embedd,
                width_embedd,
                availability_features,
            ),
            dim=-1,
        )
        rich = self.rich_mlp(rich_input)
        rich = availability.any(dim=-1, keepdim=True) * rich
        # print(torch.linalg.vector_norm(rich, dim=-1, keepdim=True)[0, ...])

        return self.layer_norm(shift_embedd + rich)


class CPeakEmbedder(nn.Module):
    def __init__(self, d_model, c_range, fourier_strategy, num_fourier_freqs):
        super().__init__()
        self.shift_embedder = ShiftEmbedder(
            *c_range,
            d_model=d_model,
            fourier_strategy=fourier_strategy,
            num_fourier_freqs=num_fourier_freqs,
        )
        self.layer_norm = nn.LayerNorm(d_model)

    def forward(self, shift: torch.Tensor):
        return self.layer_norm(self.shift_embedder(shift))


class NmrEmbedder(nn.Module):
    def __init__(
        self,
        d_model=512,
        dropout=0.1,
        fourier_strategy='log_spaced',
        h_x_min=0.01,
        h_x_max=20.0,
        c_x_min=0.1,
        c_x_max=300.0,
        num_fourier_freqs=256,
        num_multiplicities=None,
        j_min=0,
        j_max=20,
        num_rbf_centers=32,
        rbf_sigma=2,
    ):
        super().__init__()
        if num_multiplicities is None:
            raise ValueError("num_multiplicities must be provided")

        self.h_range = (h_x_min, h_x_max)
        self.c_range = (c_x_min, c_x_max)
        self.d_model = d_model
        self.dropoutp = dropout

        self.h_type_embedding = nn.Parameter(torch.empty(d_model))
        self.c_type_embedding = nn.Parameter(torch.empty(d_model))

        nn.init.normal_(self.h_type_embedding, mean=0.0, std=0.02)
        nn.init.normal_(self.c_type_embedding, mean=0.0, std=0.02)

        self.H_embedder = HPeakEmbedder(
            d_model=d_model,
            h_range=self.h_range,
            fourier_strategy=fourier_strategy,
            num_fourier_freqs=num_fourier_freqs,
            num_multiplicities=num_multiplicities,
            j_min=j_min,
            j_max=j_max,
            num_rbf_centers=num_rbf_centers,
            rbf_sigma=rbf_sigma,
        )
        self.C_embedder = CPeakEmbedder(
            d_model=d_model,
            c_range=self.c_range,
            fourier_strategy=fourier_strategy,
            num_fourier_freqs=num_fourier_freqs,
        )

    def forward(self, batch):
        h_peaks = batch["h"]
        c_peaks = batch["c"]
        h_tokens = self.H_embedder(
            shift=h_peaks["shift"],
            integration=h_peaks["integration"],
            multiplicity=h_peaks["multiplicity"],
            j_values=h_peaks["j_values"],
            width=h_peaks["range_half_span"],
            j_mask=h_peaks["j_mask"],
            availability=h_peaks["availability"],
        )
        c_tokens = self.C_embedder(
            shift=c_peaks["shift"]
        )

        h_type = self.h_type_embedding.view(1, 1, -1)
        c_type = self.c_type_embedding.view(1, 1, -1)

        h_valid = h_peaks["peak_mask"].unsqueeze(-1).to(h_tokens.dtype)
        c_valid = c_peaks["peak_mask"].unsqueeze(-1).to(c_tokens.dtype)

        h_tokens = (h_tokens + h_type) * h_valid
        c_tokens = (c_tokens + c_type) * c_valid

        nmr_peak_tokens = torch.concat([h_tokens, c_tokens], dim=1)
        nmr_peak_valid_mask = torch.concat([h_peaks["peak_mask"], c_peaks["peak_mask"]], dim=1) # True == peak is valid

        return nmr_peak_tokens, nmr_peak_valid_mask


In [ ]:
nmr_embedder = NmrEmbedder(
    d_model=512,
    dropout=0.1,
    fourier_strategy='log_spaced',     # Default strategy: lin_float_int
    h_x_min=0.01, h_x_max=20.0,
    c_x_min=0.1, c_x_max=300.0,
    num_fourier_freqs=256,      # If None, compute automatically
    num_multiplicities=len(MULTIPLICITY_TO_ID),
    j_min=0, j_max=20,
    num_rbf_centers=32,
    rbf_sigma=2,
    )

tensor([[0.0477],
        [0.0643],
        [0.0610],
        [0.0525],
        [0.0642],
        [0.0642],
        [0.0507],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0.0092],
        [0